In [1]:
import os
import pandas as pd
import numpy as np
import pathlib
import ast
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import json




def fix_dataframe(data):

    # Initialize the column 'ievent' with zeros
    data['ievent'] = 0
    # Variable to keep track of the increment for column 'ievent'
    counter = 0
    # Loop through the DataFrame rows
    for index, row in data.iterrows():
        if row['iparticle'] == 0: counter += 1
        data.at[index, 'ievent'] = counter
    #return

    return data


In [2]:
def calculate_observables(data):
    events = []
    grouped_data = data.groupby('ievent')

    for ievent, evt in grouped_data:
        # Reset or initialize variables for each event
        e_mu_minus, e_mu_plus, has_charm,has_pion, e_em, e_visible, ptx, pty, ht =0,0, 0, 0, 0, 0, 0, 0, 0
        pt_mu_minus, pt_mu_minus, phi_mu_minus, phi_mu_minus = 0, 0, 0, 0

        event_weight = 1
        # Iterate over particles in the event
        for _, row in evt.iterrows():
            truth_energy, pid, parent_pid1,weight = row.iloc[2], row.iloc[3], row.iloc[8],row.iloc[10]
            px, py, pz, e  =  row.iloc[4], row.iloc[5], row.iloc[6], row.iloc[7]
            #event_weight=1
            event_weight = abs(min(event_weight,weight))
            # Check for charm particles
            if abs(parent_pid1) in {411, 421, 431, 4122,15,521,511,531,443,5122,4232,4112,5232}:
                has_charm = 1

            if pid == -13 and  parent_pid1 in {211,321}:
                has_pion = 1


            # Update energies for mu- and mu+
            if pid == 13:
                if e>e_mu_minus:
                    e_mu_minus = e
                    pt_mu_minus = np.sqrt(px**2 + py**2)
                    phi_mu_minus = np.arctan2(py, px)
            elif pid == -13:
                if e>e_mu_plus:
                    e_mu_plus = e
                    pt_mu_plus = np.sqrt(px**2 + py**2)
                    phi_mu_plus = np.arctan2(py, px)

            # Sum EM and visible energy
            if abs(pid) in {22, 11}:
                e_em += e
            if abs(pid) not in {12, 14, 16, 39}:
                e_visible += e
                ptx += px
                pty += py
                ht += np.sqrt(px**2 + py**2)

        # Calculate missing pT
        pt_mis = np.sqrt(ptx**2 + pty**2)
                                                                                                                                                                                          
        phi_vis = np.arctan2(pty, ptx)
        phi_mis = np.arctan2(-pty, -ptx)

        # Append event observables
        events.append([ievent, truth_energy, e_mu_minus, e_mu_plus, e_em, has_charm, e_visible, pt_mis, ht, pt_mu_minus, pt_mu_plus, phi_mu_minus, phi_mu_plus, phi_mis, phi_vis, has_pion,event_weight])

    # Construct DataFrame from results
    columns = ['ievent', 'truth_energy', 'e_mu_minus', 'e_mu_plus', 'e_em', 'has_charm', 'e_visible', 'pt_mis', 'ht', 'pt_mu_minus', 'pt_mu_plus', 'phi_mu_minus', 'phi_mu_plus', 'phi_mis', 'phi_vis','has_pion','weight']
    observables = pd.DataFrame(events, columns=columns)

    return observables

#define energies
energies = [


14.21, 17.90, 22.53, 28.37, 35.71, 44.964, 56.60, 71.26, 89.71,
    112.94, 142.19, 179.00, 225.35, 283.70, 357.16, 449.64,
    566.07, 712.64, 897.16, 1129.46, 1421.90]



#define processes
processes_bg = [ "NC_nuebar_displaced","NC_numu_displaced","CC_nuebar_displaced","CC_numu_displaced"]
processes_sig = ["0.001GeV","0.01GeV","0.05GeV","0.125GeV","0.5GeV","1GeV","3.16GeV","7.94GeV","15.84GeV","16GeV","17.5GeV","18GeV","19.95GeV","21GeV","23GeV","25GeV","30GeV","40GeV"]
processes = processes_bg # + processes_sig

# Loop through all energies
for process in processes:
    for energy in energies:
        filename = f"output_events_{process}/events_new{str(energy)}.csv.zip"

        # Check if the file exists
        if not os.path.isfile(filename):
            print(f"File not found: {filename}")
            continue

        print(process, energy)
        data = pd.read_csv(filename)

        # Check if the first row has a valid value
        #if 0 not in data.index or np.isnan(data.loc[0, 'ievent']):
         #   print(f"Fixing DataFrame for {filename}")
        data = fix_dataframe(data)

        # Calculate observables and save
        observables = calculate_observables(data)
        csv_file = f"output_events_{process}/observables_new_{str(energy)}.csv.zip"
        observables.to_csv(csv_file, index=False, compression='zip')

                                                                                                                                                                                        


NC_nuebar_displaced 14.21
NC_nuebar_displaced 17.9
NC_nuebar_displaced 22.53
NC_nuebar_displaced 28.37
NC_nuebar_displaced 35.71
NC_nuebar_displaced 44.964
NC_nuebar_displaced 56.6
NC_nuebar_displaced 71.26
NC_nuebar_displaced 89.71
NC_nuebar_displaced 112.94
NC_nuebar_displaced 142.19
NC_nuebar_displaced 179.0
NC_nuebar_displaced 225.35
NC_nuebar_displaced 283.7
NC_nuebar_displaced 357.16
NC_nuebar_displaced 449.64
NC_nuebar_displaced 566.07
NC_nuebar_displaced 712.64
NC_nuebar_displaced 897.16
NC_nuebar_displaced 1129.46
NC_nuebar_displaced 1421.9
NC_numu_displaced 14.21
NC_numu_displaced 17.9
NC_numu_displaced 22.53
NC_numu_displaced 28.37
NC_numu_displaced 35.71
NC_numu_displaced 44.964
NC_numu_displaced 56.6
NC_numu_displaced 71.26
NC_numu_displaced 89.71
NC_numu_displaced 112.94
NC_numu_displaced 142.19
NC_numu_displaced 179.0
NC_numu_displaced 225.35
NC_numu_displaced 283.7
NC_numu_displaced 357.16
NC_numu_displaced 449.64
NC_numu_displaced 566.07
NC_numu_displaced 712.64
NC_num